In [1]:
!pip install -qqqq -U langchain-openai
!pip install -qqqq -U langchain-core
!pip install -qqqq -U openai
!pip install -qqqq -U pydantic
!pip install -qqqq -U langgraph

In [2]:
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.types import Command, interrupt

# Import the Python SDK for Colab secrets
from google.colab import userdata

In [3]:
import os

# Fetch the API key from Colab secrets
openai_api_key = userdata.get('OPENAI_APIKEY')

# Set up Environmental Variables
os.environ["OPENAI_API_KEY"] = openai_api_key

In [4]:
# ==========================================
# 1. DEFINE TOOLS WITH THE @tool DECORATOR
# ==========================================

@tool
def write_file(filename: str, content: str) -> str:
    """Write content to a file. Requires human approval before execution."""
    # Simulating file write operation
    return f"Successfully wrote content to {filename}"

@tool
def read_file(filename: str) -> str:
    """Read contents of a file. Safe to run automatically without review."""
    return f"Contents of {filename}: [Sample file data line 1, line 2]"

tools_map = {
    "write_file": write_file,
    "read_file": read_file
}

In [5]:
# ==========================================
# 2. BIND TOOLS TO CHATOPENAI
# ==========================================

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(list(tools_map.values()))

In [27]:
# ==========================================
# 3. THE HITL CONTROL LOOP
# ==========================================

def run_hitl_agent(user_prompt: str):
    messages = [HumanMessage(content=user_prompt)]
    print(f"\nUser Prompt: {user_prompt}\n")

    # Step A: First LLM turn to see if a tool is requested
    response = llm_with_tools.invoke(messages)
    messages.append(response)

    # Step B: Check if the model triggered any tool calls
    if response.tool_calls:
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_call_id = tool_call["id"]
            print(f"\n--- Tool Call: {tool_name} ---")
            print(f"Arguments: {tool_args}")
            # Sensitive tools require HITL interruption
            if tool_name == "write_file":
                print(f"\n--- 🛑 HITL PAUSE: Sensitive Action Requested ---")
                print(f"Tool: {tool_name}")
                print(f"Arguments: {tool_args}")

                # Prompt the human for approval
                choice = input("Do you approve this action? (y/n): ").strip().lower()

                if choice == 'y':
                    print(f"--- [Approved] Executing {tool_name} ---")
                    tool_result = tools_map[tool_name].invoke(tool_args)
                    messages.append(ToolMessage(content=tool_result, tool_call_id=tool_call_id))
                elif choice == 'n':
                    print(f"--- [Rejected] Action denied by human ---")
                    # messages[0].content = "You have rejected the execution"
                    messages[0] = HumanMessage(content="You have rejected the execution")
                    messages.append(ToolMessage(content=messages[0].content, tool_call_id=tool_call_id))
                else:
                   print(f"--- [Wrong] Choice by human ---")
                   rejection_msg = "Error: Wrong choice send ."
                   messages[0].content = "You should have choosen either y or n "
                   messages.append(ToolMessage(content=messages[0].content, tool_call_id=tool_call_id))
            elif tool_name == "read_file":
              # Safe tools execute automatically without pausing
              print(f"--- [Auto-Executing Safe Tool] {tool_name} ---")
              tool_result = tools_map[tool_name].invoke(tool_args)
              messages.append(ToolMessage(content=tool_result, tool_call_id=tool_call_id))
            else :
              print(f"--- [Unhandled Tool Call] {tool_name} ---")
        # Step C: Send the tool response back to the LLM for final synthesis
        print("\n--- Sending tool results back to LLM ---")
        print(f"Messages: {messages}")
        final_response = llm_with_tools.invoke(messages)
        return final_response.content
    else:
        # If no tools were called, return the text response directly
        return response.content

In [28]:
# ==========================================
# 3. EXECUTION
# ==========================================

# This prompt will trigger 'write_file', which activates the HITL prompt pause
final_output = run_hitl_agent("Write a welcome message to output.txt")

print("\n==============================")
print("FINAL AGENT RESPONSE:")
print("==============================\n")
print(final_output)


User Prompt: Write a welcome message to output.txt


--- Tool Call: write_file ---
Arguments: {'filename': 'output.txt', 'content': 'Welcome to our application! We are glad to have you here. Enjoy your experience!'}

--- 🛑 HITL PAUSE: Sensitive Action Requested ---
Tool: write_file
Arguments: {'filename': 'output.txt', 'content': 'Welcome to our application! We are glad to have you here. Enjoy your experience!'}
Do you approve this action? (y/n): n
--- [Rejected] Action denied by human ---

--- Sending tool results back to LLM ---
Messages: [HumanMessage(content='You have rejected the execution', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 87, 'total_tokens': 122, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'au